# 06 — Population Data Preparation

## Objective

This notebook validates and prepares the district level population dataset for integration with the monthly dengue and weather datasets.

The main objectives are to:

1. validate the standardized population dataset;
2. investigate duplicate population records;
3. distinguish identical duplicate records from conflicting district-year observations;
4. verify geographic and temporal coverage;
5. prepare one validated population value per administrative district and year;
6. examine the mismatch between population coverage (2014–2024) and dengue/weather coverage (2010–2024);
7. construct monthly population features only after the annual data have been validated.

Raw population data are not modified in this notebook.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
print("Project root:", PROJECT_ROOT)
print("Interim directory:", INTERIM_DIR)

Project root: d:\Dengue_Project
Interim directory: d:\Dengue_Project\data\interim


In [3]:
population_files = list(
    INTERIM_DIR.glob(
        "*population*geography*standardized*.csv"
    )
)
population_files

[WindowsPath('d:/Dengue_Project/data/interim/population_geography_standardized.csv')]

In [4]:
if len(population_files) != 1:
    raise ValueError(
        f"Expected exactly 1 standardized population file, "
        f"but found {len(population_files)}: {population_files}"
    )
population_path = population_files[0]
print("Population file:", population_path)

Population file: d:\Dengue_Project\data\interim\population_geography_standardized.csv


In [5]:
population = pd.read_csv(
    population_path
)
print(
    "Population shape:",
    population.shape
)
population.head()

Population shape: (475, 5)


,district,year,population,source_area,administrative_district
0,Colombo,2014,2353000,Colombo,Colombo
1,Colombo,2015,2367000,Colombo,Colombo
2,Colombo,2016,2387000,Colombo,Colombo
3,Colombo,2017,2407000,Colombo,Colombo
4,Gampaha,2014,2339000,Gampaha,Gampaha


In [6]:
population[
    [
        "year",
        "population"
    ]
].describe()

,year,population
count,475.000000,4.750000e+02
mean,2018.368421,8.632632e+05
std,2.926860,6.077077e+05
min,2014.000000,9.500000e+04
25%,2016.000000,4.830000e+05
50%,2018.000000,7.630000e+05
75%,2021.000000,1.127500e+06
max,2024.000000,2.480000e+06


In [7]:
population.isna().sum()

district                   0
year                       0
population                 0
source_area                0
administrative_district    0
dtype: int64

In [8]:
print(
    "Total missing values:",
    population.isna().sum().sum()
)

Total missing values: 0


In [9]:
print(
    "Minimum population:",
    population["population"].min()
)
print(
    "Maximum population:",
    population["population"].max()
)
print(
    "Negative population values:",
    (
        population["population"] < 0
    ).sum()
)
print(
    "Zero population values:",
    (
        population["population"] == 0
    ).sum()
)

Minimum population: 95000
Maximum population: 2480000
Negative population values: 0
Zero population values: 0


Temporal Coverage

In [10]:
population_years = sorted(
    population["year"].unique()
)
print(
    "Available years:",
    population_years
)
print(
    "First year:",
    population["year"].min()
)
print(
    "Last year:",
    population["year"].max()
)
print(
    "Number of unique years:",
    population["year"].nunique()
)

Available years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
First year: 2014
Last year: 2024
Number of unique years: 11


In [11]:
expected_years = list(
    range(
        population["year"].min(),
        population["year"].max() + 1
    )
)
missing_years_within_coverage = sorted(
    set(expected_years)
    - set(population_years)
)
print(
    "Missing years within available coverage:",
    missing_years_within_coverage
)

Missing years within available coverage: []


Geographic Coverage

In [12]:
population_districts = sorted(
    population[
        "administrative_district"
    ].unique()
)
print(
    "Administrative districts:",
    len(population_districts)
)
population_districts

Administrative districts: 25


['Ampara',
 'Anuradhapura',
 'Badulla',
 'Batticaloa',
 'Colombo',
 'Galle',
 'Gampaha',
 'Hambantota',
 'Jaffna',
 'Kalutara',
 'Kandy',
 'Kegalle',
 'Kilinochchi',
 'Kurunegala',
 'Mannar',
 'Matale',
 'Matara',
 'Monaragala',
 'Mullaitivu',
 'Nuwara Eliya',
 'Polonnaruwa',
 'Puttalam',
 'Ratnapura',
 'Trincomalee',
 'Vavuniya']

In [13]:
years_per_district = (
    population
    .groupby(
        "administrative_district"
    )["year"]
    .nunique()
    .sort_values()
)
years_per_district

administrative_district
Ampara          11
Anuradhapura    11
Badulla         11
Batticaloa      11
Colombo         11
Galle           11
Gampaha         11
Hambantota      11
Jaffna          11
Kalutara        11
Kandy           11
Kegalle         11
Kilinochchi     11
Kurunegala      11
Mannar          11
Matale          11
Matara          11
Monaragala      11
Mullaitivu      11
Nuwara Eliya    11
Polonnaruwa     11
Puttalam        11
Ratnapura       11
Trincomalee     11
Vavuniya        11
Name: year, dtype: int64

In [14]:
exact_duplicate_count = (
    population
    .duplicated()
    .sum()
)
print(
    "Exact duplicate rows:",
    exact_duplicate_count
)
print(
    "Rows before removing duplicates:",
    len(population)
)
print(
    "Rows after exact deduplication:",
    len(
        population.drop_duplicates()
    )
)

Exact duplicate rows: 200
Rows before removing duplicates: 475
Rows after exact deduplication: 275


In [15]:
duplicate_district_year_rows = (
    population
    .duplicated(
        subset=[
            "administrative_district",
            "year"
        ],
        keep=False
    )
)
print(
    "Rows belonging to repeated district-year keys:",
    duplicate_district_year_rows.sum()
)

Rows belonging to repeated district-year keys: 400


In [16]:
population[
    duplicate_district_year_rows
].sort_values(
    [
        "administrative_district",
        "year"
    ]
).head(40)

,district,year,population,source_area,administrative_district
60,Ampara,2014,666000,Ampara,Ampara
335,Ampara,2014,666000,Ampara,Ampara
61,Ampara,2015,674000,Ampara,Ampara
336,Ampara,2015,674000,Ampara,Ampara
62,Ampara,2016,686000,Ampara,Ampara
337,Ampara,2016,686000,Ampara,Ampara
63,Ampara,2017,700000,Ampara,Ampara
338,Ampara,2017,700000,Ampara,Ampara
160,Ampara,2018,719000,Ampara,Ampara
435,Ampara,2018,719000,Ampara,Ampara


conflicting population values

In [17]:
population_values_per_key = (
    population
    .groupby(
        [
            "administrative_district",
            "year"
        ]
    )["population"]
    .nunique()
)
conflicting_keys = (
    population_values_per_key[
        population_values_per_key > 1
    ]
)
print(
    "District-year keys with conflicting population values:",
    len(conflicting_keys)
)
conflicting_keys.head(20)

District-year keys with conflicting population values: 0


Series([], Name: population, dtype: int64)

Check unique district-year count

In [18]:
unique_district_years = (
    population[
        [
            "administrative_district",
            "year"
        ]
    ]
    .drop_duplicates()
)
print(
    "Unique district-year combinations:",
    len(unique_district_years)
)
expected_combinations = (
    population[
        "administrative_district"
    ].nunique()
    *
    population[
        "year"
    ].nunique()
)
print(
    "Expected combinations:",
    expected_combinations
)
print(
    "Complete district-year coverage:",
    len(unique_district_years)
    == expected_combinations
)

Unique district-year combinations: 275
Expected combinations: 275
Complete district-year coverage: True


## Duplicate Record Assessment

The standardized population dataset contains 475 rows, of which 200 are exact duplicate rows.

After removing exact duplicates, 275 unique observations remain. This corresponds exactly to the expected 25 administrative districts × 11 years (2014–2024).

A district year consistency check found no cases where the same administrative district and year were associated with different population values. Therefore, the duplicated observations represent repeated identical records rather than conflicting population estimates.

Exact duplicates can consequently be removed without changing the population information contained in the dataset.

Safely remove exact duplicates

In [19]:
population_clean = (
    population
    .drop_duplicates()
    .copy()
    .reset_index(drop=True)
)
print(
    "Rows before deduplication:",
    len(population)
)
print(
    "Rows after deduplication:",
    len(population_clean)
)
print(
    "Rows removed:",
    len(population) - len(population_clean)
)

Rows before deduplication: 475
Rows after deduplication: 275
Rows removed: 200


In [20]:
print(
    "Exact duplicates remaining:",
    population_clean.duplicated().sum()
)
print(
    "Duplicate district-year keys remaining:",
    population_clean.duplicated(
        subset=[
            "administrative_district",
            "year"
        ]
    ).sum()
)

Exact duplicates remaining: 0
Duplicate district-year keys remaining: 0


Confirm complete coverage after deduplication

In [21]:
clean_districts = (
    population_clean[
        "administrative_district"
    ].nunique()
)
clean_years = (
    population_clean[
        "year"
    ].nunique()
)
expected_clean_rows = (
    clean_districts * clean_years
)
print(
    "Administrative districts:",
    clean_districts
)
print(
    "Years:",
    clean_years
)
print(
    "Expected rows:",
    expected_clean_rows
)
print(
    "Actual rows:",
    len(population_clean)
)
print(
    "Complete district-year coverage:",
    len(population_clean)
    == expected_clean_rows
)

Administrative districts: 25
Years: 11
Expected rows: 275
Actual rows: 275
Complete district-year coverage: True


Check population trend behavior

In [22]:
population_clean = (
    population_clean
    .sort_values(
        [
            "administrative_district",
            "year"
        ]
    )
    .reset_index(drop=True)
)

In [23]:
population_clean[
    "annual_population_change"
] = (
    population_clean
    .groupby(
        "administrative_district"
    )["population"]
    .diff()
)

In [24]:
population_decreases = (
    population_clean[
        population_clean[
            "annual_population_change"
        ] < 0
    ]
)
print(
    "District-year observations with population decrease:",
    len(population_decreases)
)
population_decreases[
    [
        "administrative_district",
        "year",
        "population",
        "annual_population_change"
    ]
]

District-year observations with population decrease: 44


,administrative_district,year,population,annual_population_change
9,Ampara,2023,749000,-5000.0
10,Ampara,2024,746000,-3000.0
20,Anuradhapura,2023,950000,-7000.0
21,Anuradhapura,2024,944000,-6000.0
31,Badulla,2023,896000,-3000.0
32,Badulla,2024,895000,-1000.0
42,Batticaloa,2023,582000,-8000.0
43,Batticaloa,2024,577000,-5000.0
52,Colombo,2022,2478000,-2000.0
53,Colombo,2023,2460000,-18000.0


Calculate annual percentage change

In [25]:
population_clean[
    "annual_population_change_percent"
] = (
    population_clean
    .groupby(
        "administrative_district"
    )["population"]
    .pct_change()
    * 100
)
population_clean[
    [
        "administrative_district",
        "year",
        "population",
        "annual_population_change",
        "annual_population_change_percent"
    ]
].head(15)

,administrative_district,year,population,annual_population_change,annual_population_change_percent
0,Ampara,2014,666000,NaN,NaN
1,Ampara,2015,674000,8000.0,1.201201
2,Ampara,2016,686000,12000.0,1.780415
3,Ampara,2017,700000,14000.0,2.040816
4,Ampara,2018,719000,19000.0,2.714286
5,Ampara,2019,728000,9000.0,1.251739
6,Ampara,2020,736000,8000.0,1.098901
7,Ampara,2021,752000,16000.0,2.173913
8,Ampara,2022,754000,2000.0,0.265957
9,Ampara,2023,749000,-5000.0,-0.663130


In [26]:
population_clean[
    "annual_population_change_percent"
].describe()

count    250.000000
mean       0.694047
std        0.807475
min       -2.040816
25%        0.255565
50%        0.821590
75%        1.128668
max        3.278689
Name: annual_population_change_percent, dtype: float64

Find largest increases and decreases

In [27]:
population_clean[
    [
        "administrative_district",
        "year",
        "population",
        "annual_population_change_percent"
    ]
].sort_values(
    "annual_population_change_percent"
).head(10)

,administrative_district,year,population,annual_population_change_percent
202,Mullaitivu,2018,96000,-2.040816
42,Batticaloa,2023,582000,-1.355932
119,Kandy,2023,1482000,-1.134089
108,Kalutara,2023,1279000,-1.006192
175,Matale,2024,520000,-0.952381
174,Matale,2023,525000,-0.943396
109,Kalutara,2024,1267000,-0.938233
229,Polonnaruwa,2023,445000,-0.890869
153,Kurunegala,2024,1712000,-0.868558
152,Kurunegala,2023,1727000,-0.861079


In [28]:
population_clean[
    [
        "administrative_district",
        "year",
        "population",
        "annual_population_change_percent"
    ]
].sort_values(
    "annual_population_change_percent",
    ascending=False
).head(10)

,administrative_district,year,population,annual_population_change_percent
136,Kilinochchi,2018,126000,3.278689
158,Mannar,2018,109000,2.830189
4,Ampara,2018,719000,2.714286
257,Trincomalee,2018,421000,2.433090
137,Kilinochchi,2019,129000,2.380952
260,Trincomalee,2021,441000,2.320186
139,Kilinochchi,2021,133000,2.307692
267,Vavuniya,2017,187000,2.185792
7,Ampara,2021,752000,2.173913
3,Ampara,2017,700000,2.040816


## Population Temporal Coverage

The validated population dataset provides complete annual district level population values from 2014 through 2024.

However, the dengue and weather datasets cover 2010 through 2024. Therefore, population observations are unavailable for 2010–2013.

These earlier population values will not be created through arbitrary backward filling or extrapolation in this notebook. The absence of population data for 2010–2013 represents a source-coverage limitation rather than isolated missing observations within the available population series.

The validated 2014–2024 population observations are retained, and the effect of this temporal mismatch will be addressed explicitly during dataset integration and modeling.

Decide annual → monthly representation

## Annual-to-Monthly Population Representation

The population dataset contains annual district-level estimates, whereas the dengue forecasting dataset uses monthly observations.

For integration, each annual population estimate is assigned to all twelve months of the corresponding calendar year.

For example, the population estimate for a district in 2018 is used for January through December 2018.

This transformation does not create new population estimates between years. It converts the temporal representation of the existing annual estimate so that it can be joined with the monthly dengue and weather datasets.

No population values are generated for 2010–2013 because those years are outside the coverage of the available population source.

Keep only required columns

In [29]:
population_annual = (
    population_clean[
        [
            "administrative_district",
            "year",
            "population"
        ]
    ]
    .copy()
)

population_annual.head()

,administrative_district,year,population
0,Ampara,2014,666000
1,Ampara,2015,674000
2,Ampara,2016,686000
3,Ampara,2017,700000
4,Ampara,2018,719000


In [30]:
print(
    "Annual population shape:",
    population_annual.shape
)
print(
    "Districts:",
    population_annual[
        "administrative_district"
    ].nunique()
)
print(
    "Years:",
    population_annual[
        "year"
    ].nunique()
)

Annual population shape: (275, 3)
Districts: 25
Years: 11


In [31]:
monthly_population_records = []
for _, row in population_annual.iterrows():
    for month_number in range(1, 13):
        monthly_population_records.append({
            "date": pd.Timestamp(
                year=int(row["year"]),
                month=month_number,
                day=1
            ),
            "year": int(row["year"]),
            "month_number": month_number,
            "administrative_district":
                row["administrative_district"],
            "population":
                row["population"]
        })
population_monthly = pd.DataFrame(
    monthly_population_records
)
population_monthly.head(15)

,date,year,month_number,administrative_district,population
0,2014-01-01,2014,1,Ampara,666000
1,2014-02-01,2014,2,Ampara,666000
2,2014-03-01,2014,3,Ampara,666000
3,2014-04-01,2014,4,Ampara,666000
4,2014-05-01,2014,5,Ampara,666000
5,2014-06-01,2014,6,Ampara,666000
6,2014-07-01,2014,7,Ampara,666000
7,2014-08-01,2014,8,Ampara,666000
8,2014-09-01,2014,9,Ampara,666000
9,2014-10-01,2014,10,Ampara,666000


Check expected monthly size

In [32]:
expected_monthly_rows = (
    25 * 11 * 12
)
print(
    "Expected monthly rows:",
    expected_monthly_rows
)
print(
    "Actual monthly rows:",
    len(population_monthly)
)
print(
    "Correct row count:",
    len(population_monthly)
    == expected_monthly_rows
)

Expected monthly rows: 3300
Actual monthly rows: 3300
Correct row count: True


Check monthly date range

In [33]:
print(
    "First monthly date:",
    population_monthly[
        "date"
    ].min()
)
print(
    "Last monthly date:",
    population_monthly[
        "date"
    ].max()
)
print(
    "Unique months:",
    population_monthly[
        "date"
    ].nunique()
)

First monthly date: 2014-01-01 00:00:00
Last monthly date: 2024-12-01 00:00:00
Unique months: 132


Check months per district

In [34]:
months_per_district = (
    population_monthly
    .groupby(
        "administrative_district"
    )["date"]
    .nunique()
)
print(
    "Minimum months per district:",
    months_per_district.min()
)
print(
    "Maximum months per district:",
    months_per_district.max()
)

Minimum months per district: 132
Maximum months per district: 132


Check duplicate district-month keys

In [35]:
duplicate_monthly_keys = (
    population_monthly
    .duplicated(
        subset=[
            "administrative_district",
            "date"
        ]
    )
    .sum()
)
print(
    "Duplicate district-month keys:",
    duplicate_monthly_keys
)

Duplicate district-month keys: 0


Check missing monthly values

In [36]:
population_monthly.isna().sum()
print(
    "Total missing values:",
    population_monthly.isna().sum().sum()
)

Total missing values: 0


Validate annual values were repeated correctly

In [37]:
monthly_values_per_year = (
    population_monthly
    .groupby(
        [
            "administrative_district",
            "year"
        ]
    )["population"]
    .nunique()
)
print(
    "Maximum unique population values "
    "within a district-year:",
    monthly_values_per_year.max()
)
print(
    "District-years with more than one "
    "monthly population value:",
    (
        monthly_values_per_year > 1
    ).sum()
)

Maximum unique population values within a district-year: 1
District-years with more than one monthly population value: 0


Compare monthly values back to annual source

In [38]:
monthly_back_to_annual = (
    population_monthly
    .groupby(
        [
            "administrative_district",
            "year"
        ],
        as_index=False
    )["population"]
    .first()
)
population_comparison = (
    population_annual
    .merge(
        monthly_back_to_annual,
        on=[
            "administrative_district",
            "year"
        ],
        suffixes=(
            "_annual",
            "_monthly"
        )
    )
)
population_values_preserved = (
    population_comparison[
        "population_annual"
    ]
    ==
    population_comparison[
        "population_monthly"
    ]
).all()

print(
    "Annual population values preserved:",
    population_values_preserved
)

Annual population values preserved: True


In [39]:
population_validation = pd.DataFrame({
    "Validation Check": [
        "Administrative districts",
        "Annual years",
        "Unique annual district-year records",
        "Exact duplicates after cleaning",
        "Duplicate district-year keys after cleaning",
        "Monthly observations",
        "Unique monthly dates",
        "Minimum months per district",
        "Maximum months per district",
        "Duplicate district-month keys",
        "Missing monthly values",
        "Annual population values preserved"
    ],
    "Result": [
        population_annual[
            "administrative_district"
        ].nunique(),
        population_annual[
            "year"
        ].nunique(),
        len(population_annual),
        population_clean
        .duplicated()
        .sum(),
        population_clean
        .duplicated(
            subset=[
                "administrative_district",
                "year"
            ]
        )
        .sum(),
        len(population_monthly),
        population_monthly[
            "date"
        ].nunique(),
        months_per_district.min(),
        months_per_district.max(),
        duplicate_monthly_keys,
        population_monthly
        .isna()
        .sum()
        .sum(),

        population_values_preserved
    ]
})

population_validation

,Validation Check,Result
0,Administrative districts,25
1,Annual years,11
2,Unique annual district-year records,275
3,Exact duplicates after cleaning,0
4,Duplicate district-year keys after cleaning,0
5,Monthly observations,3300
6,Unique monthly dates,132
7,Minimum months per district,132
8,Maximum months per district,132
9,Duplicate district-month keys,0


## Population Trend Assessment

Annual population changes were examined after duplicate removal.

A total of 44 district year observations showed a decrease relative to the preceding year. The observed annual percentage changes ranged from approximately -2.04% to +3.28%.

Population is not assumed to increase monotonically. Changes may reflect demographic variation and/or the estimation methodology of the source data. Therefore, population decreases were not automatically treated as errors or modified.

The original validated annual population estimates are retained unchanged for subsequent analysis.

In [40]:
population_monthly_final = (
    population_monthly[
        [
            "date",
            "year",
            "month_number",
            "administrative_district",
            "population"
        ]
    ]
    .sort_values(
        [
            "administrative_district",
            "date"
        ]
    )
    .reset_index(drop=True)
)

population_monthly_final.head()

,date,year,month_number,administrative_district,population
0,2014-01-01,2014,1,Ampara,666000
1,2014-02-01,2014,2,Ampara,666000
2,2014-03-01,2014,3,Ampara,666000
3,2014-04-01,2014,4,Ampara,666000
4,2014-05-01,2014,5,Ampara,666000


In [41]:
print(
    "Final shape:",
    population_monthly_final.shape
)
print(
    "Administrative districts:",
    population_monthly_final[
        "administrative_district"
    ].nunique()
)
print(
    "Unique months:",
    population_monthly_final[
        "date"
    ].nunique()
)
print(
    "Date range:",
    population_monthly_final["date"].min(),
    "to",
    population_monthly_final["date"].max()
)
print(
    "Missing values:",
    population_monthly_final
    .isna()
    .sum()
    .sum()
)
print(
    "Duplicate district-month keys:",
    population_monthly_final
    .duplicated(
        subset=[
            "administrative_district",
            "date"
        ]
    )
    .sum()
)

Final shape: (3300, 5)
Administrative districts: 25
Unique months: 132
Date range: 2014-01-01 00:00:00 to 2024-12-01 00:00:00
Missing values: 0
Duplicate district-month keys: 0


In [42]:
population_output = (
    INTERIM_DIR /
    "population_monthly_prepared_2014_2024.csv"
)
population_monthly_final.to_csv(
    population_output,
    index=False
)
print(
    "Saved to:",
    population_output
)
print(
    "File exists:",
    population_output.exists()
)

Saved to: d:\Dengue_Project\data\interim\population_monthly_prepared_2014_2024.csv
File exists: True


In [43]:
population_check = pd.read_csv(
    population_output,
    parse_dates=["date"]
)
print(
    "Reloaded shape:",
    population_check.shape
)
print(
    "Reloaded districts:",
    population_check[
        "administrative_district"
    ].nunique()
)
print(
    "Reloaded months:",
    population_check[
        "date"
    ].nunique()
)
print(
    "Missing values:",
    population_check
    .isna()
    .sum()
    .sum()
)
print(
    "Duplicate district-month keys:",
    population_check
    .duplicated(
        subset=[
            "administrative_district",
            "date"
        ]
    )
    .sum()
)

Reloaded shape: (3300, 5)
Reloaded districts: 25
Reloaded months: 132
Missing values: 0
Duplicate district-month keys: 0


## Explicitly record the 2010–2013 coverage gap

In [44]:
target_years = list(
    range(2010, 2025)
)
population_coverage = pd.DataFrame({
    "year": target_years
})
population_coverage[
    "population_available"
] = (
    population_coverage["year"]
    .isin(
        population_annual[
            "year"
        ].unique()
    )
)

population_coverage

,year,population_available
0,2010,False
1,2011,False
2,2012,False
3,2013,False
4,2014,True
5,2015,True
6,2016,True
7,2017,True
8,2018,True
9,2019,True


## Key Findings

1. The standardized population dataset initially contained 475 rows.

2. A total of 200 rows were exact duplicates.

3. No conflicting population values were identified for repeated administrative-district and year combinations.

4. After removing exact duplicates, 275 unique district year observations remained, exactly matching the expected 25 administrative districts × 11 years.

5. All 25 administrative districts have complete annual population coverage from 2014 through 2024.

6. No missing, zero or negative population values were identified.

7. Annual population changes were examined rather than assuming that population must increase every year. A total of 44 district year observations showed decreases and these values were retained unchanged because a decrease alone does not demonstrate a data error.

8. Annual population estimates were converted to monthly temporal resolution by assigning each annual estimate to all twelve months of its corresponding year.

9. The monthly dataset contains 3,300 observations representing 25 administrative districts across 132 months from January 2014 through December 2024.

10. No duplicate district month keys or missing values were introduced during monthly expansion and the original annual population values were preserved.

11. Population observations are unavailable for 2010–2013. These years are outside the temporal coverage of the available population source and are therefore treated as a source coverage limitation rather than ordinary missing observations.

12. No population values for 2010–2013 were generated through backward filling, interpolation or extrapolation.

## Conclusion

The district level population dataset was successfully validated and prepared for integration with the monthly dengue and weather datasets.

The original standardized dataset contained 200 exact duplicate rows. Investigation confirmed that repeated district year records did not contain conflicting population values. Exact duplicates were therefore safely removed, producing 275 unique annual observations covering all 25 administrative districts from 2014 through 2024.

Annual population trends were also examined. Population decreases were observed in several district year records; however, these observations were not automatically classified as errors or modified because population is not required to increase monotonically.

For compatibility with the monthly forecasting dataset, each validated annual population estimate was assigned to all twelve months of its corresponding year. This produced 3,300 district month observations covering January 2014 through December 2024. The transformation introduced no missing values or duplicate district month keys and preserved the original annual population estimates.

Population data are unavailable for 2010–2013, whereas the dengue and weather datasets begin in 2010. No artificial population estimates were generated for these years. This temporal coverage mismatch is retained explicitly and will be addressed during subsequent dataset integration and modeling decisions.

No raw population data were modified during this preparation process.